# 03 — Waypoints (API_TASK)
Controle manual do braço via `PUT /waypoints`.

## Pré-requisito
```bash
cd managing && python main.py
```

## Formato
| Tipo | Body |
|---|---|
| Waypoint único   | `{"waypoints": [x, y, z, gripper]}` |
| Sequência        | `{"waypoints": [[x,y,z,g], [x,y,z,g], ...]}` |

`gripper`: `1.0` = aberta · `-1.0` = fechada

In [ ]:
import requests
import json
import time

BASE_URL = "http://localhost:8000"

def get(endpoint: str) -> dict:
    r = requests.get(f"{BASE_URL}{endpoint}", timeout=3)
    r.raise_for_status()
    return r.json()

def put(endpoint: str, body: dict) -> dict:
    r = requests.put(f"{BASE_URL}{endpoint}", json=body, timeout=3)
    r.raise_for_status()
    return r.json()

def pp(data: dict):
    print(json.dumps(data, indent=2, default=str))

print("Setup OK")

---
## Passo 1 — Ativar API_TASK
Sempre ative a `API_TASK` antes de enviar waypoints.

In [ ]:
put("/task", {"strategy": "API_TASK"})
print("API_TASK ativada.")

---
## Waypoint único
Move o end-effector para uma posição `[x, y, z]` com gripper definido.

In [ ]:
response = put("/waypoints", {"waypoints": [0.1, 0.0, 0.3, 1.0]})
print(response)

---
## Sequência de waypoints
O braço executa cada waypoint em ordem, avançando quando chega perto o suficiente.

In [ ]:
response = put("/waypoints", {"waypoints": [
    [ 0.1,  0.0,  0.4,  1.0],   # subir com garra aberta
    [ 0.1,  0.0,  0.2,  1.0],   # descer
    [ 0.1,  0.0,  0.2, -1.0],   # fechar garra
    [ 0.1,  0.0,  0.4, -1.0],   # subir com garra fechada
]})
print(response)

---
## Exemplo: pegar o cubo e levar ao target
Ajuste as posições conforme o estado atual do simulador (`GET /perception`).

In [ ]:
p = get("/perception")
cx, cy, cz = p["cube_position"]
tx, ty, tz = p["target_position"]

put("/task", {"strategy": "API_TASK"})

response = put("/waypoints", {"waypoints": [
    [cx,  cy,  cz + 0.15,  1.0],   # posicionar acima do cubo
    [cx,  cy,  cz + 0.03,  1.0],   # descer ate o cubo
    [cx,  cy,  cz + 0.03, -1.0],   # fechar garra
    [cx,  cy,  cz + 0.15, -1.0],   # subir com cubo
    [tx,  ty,  tz + 0.15, -1.0],   # mover para cima do target
    [tx,  ty,  tz + 0.03, -1.0],   # descer ate o target
    [tx,  ty,  tz + 0.03,  1.0],   # soltar
]})
print(response)